# Demo: Event Summarization using SmolVLM

This notebook provides a demonstration for the second task of the assignment. Given the strong performance of vision-language models across a variety of tasks, we use an efficient, lightweight model: [SmolVLM 2](https://huggingface.co/blog/smolvlm2)
The goal of this notebook is to guide you on how to: design prompts for event detection, process model outputs, and evaluate the quality of detected events.
You are expected to build upon this baseline and critically evaluate your chosen approaches.

In [ ]:
!pip install num2words

In [ ]:
!pip install av


In [ ]:
!pip install transformers accelerate sentence-transformers av opencv-python
!pip install transformers open_clip_torch

SmolVLM2 is a lightweight vision-language model designed for efficient video understanding with relatively few parameters. The material presented here is largely based on the original documentation and is intended to provide a starting point for developing your own approach.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import numpy as np
from sentence_transformers import SentenceTransformer, util

# Load the lightweight VLM (SmolVLM2)
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,  #to save memory
).to("cuda")

# Load a sentence transformer for the embedding similarity check
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

Annotation paths are hardcoded for video 20 and may be changed for the other videos.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer, util
import numpy as np
import os

def load_manual_annotations(file_path):
    descriptions = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip():
                parts = line.split(',')
                descriptions.append(parts[0].strip())
    return descriptions

def load_annotations_with_scores(file_path):
    """
    Loads annotations returning description and subjectivity score.
    Returns list of dicts: {description, score}
    """
    annotations = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 3:
                continue
            description = parts[0].strip()
            try:
                score = int(parts[2].strip())
            except ValueError:
                score = 0
            annotations.append({'description': description, 'score': score})
    return annotations

def compute_cohen_kappa(ann_path_1, ann_path_2, sim_threshold=0.40):
    """
    Computes Cohen's Kappa between two annotators for one video.
    Strategy:
      - Load both annotation files with descriptions and subjectivity scores
      - Match events across annotators using embedding similarity
      - Only keep matched pairs above sim_threshold (genuinely same event)
      - Compute Kappa on the subjectivity scores of matched pairs
    Returns: kappa score (float), number of matched pairs (int)
    """
    anns1 = load_annotations_with_scores(ann_path_1)
    anns2 = load_annotations_with_scores(ann_path_2)
    if not anns1 or not anns2:
        print('  WARNING: one or both annotation files empty.')
        return None, 0
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    descs1 = [a['description'] for a in anns1]
    descs2 = [a['description'] for a in anns2]
    embs1 = embed_model.encode(descs1, convert_to_tensor=True, normalize_embeddings=True)
    embs2 = embed_model.encode(descs2, convert_to_tensor=True, normalize_embeddings=True)
    sim   = util.cos_sim(embs1, embs2)
    scores1, scores2 = [], []
    used = set()
    for i in range(len(anns1)):
        best_j   = int(sim[i].argmax())
        best_sim = float(sim[i][best_j])
        if best_sim >= sim_threshold and best_j not in used:
            scores1.append(anns1[i]['score'])
            scores2.append(anns2[best_j]['score'])
            used.add(best_j)
    if len(scores1) < 2:
        print(f'  WARNING: only {len(scores1)} matched pairs — Kappa unreliable.')
        return None, len(scores1)
    kappa = cohen_kappa_score(scores1, scores2)
    return kappa, len(scores1)

# Load annotator 1 for single-video run
ANNOTATION_PATH_1 = "/content/18-SumMe-video_21-annotator1.txt"
ANNOTATION_PATH_2 = "/content/18-SumMe-video_21-annotator2.txt"

manual_annotations = load_manual_annotations(ANNOTATION_PATH_1)
print(f'Loaded {len(manual_annotations)} annotations.')

if os.path.exists(ANNOTATION_PATH_2):
    kappa, n_matched = compute_cohen_kappa(ANNOTATION_PATH_1, ANNOTATION_PATH_2)
    if kappa is not None:
        print(f"Cohen's Kappa (Ann1 vs Ann2): {kappa:.3f} ({n_matched} matched pairs)")
else:
    print('Annotator 2 file not found — skipping Kappa for single-video test.')

The video path below may be changed for each different video.

In [ ]:
video_path = "/content/video_21.mp4"

Since SmolVLM2 is a vision-language model (VLM), you will need to design an appropriate prompt for the task. The prompt can be flexible and should be adapted based on the type of output you want (e.g., event lists, descriptions, or timestamps).

Vision-language models process visual inputs (such as frames or videos) together with text by encoding them into a shared representation space. The exact input format may vary across models.

In our case, the HuggingFace implementation handles most of the input preprocessing (e.g., frame sampling and formatting), allowing you to focus primarily on prompt design and output parsing.

The detected events are often highly descriptive, but they do not always capture the most important or salient moments in the video.

To address this, you may explore alternative strategies. One approach is to use a language model (LLM) to process frame-level descriptions generated from the video.

For example, you can experiment with randomly sampling frames from the video, generating descriptions for each frame using a vision-language model (VLM), and then aggregating these descriptions into a structured set of events using an LLM.


In [ ]:

import cv2, numpy as np
from PIL import Image

def sample_frames(video_path, num_frames=24):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    if total <= 0:
        raise ValueError("Video has no frames.")
    idxs = np.linspace(0, total-1, min(num_frames,total), dtype=int)
    frames=[]; meta=[]
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
            meta.append({"frame":int(idx), "time":float(idx/fps)})
    cap.release()
    return frames, meta


You may improve event detection by splitting the video into smaller temporal segments and prompting the VLM on each segment separately. The resulting partial event descriptions can then be combined into a single, coherent list using an LLM.

Crucially, you must define an evaluation scheme for event coverage, for example:
-How many annotated events are correctly detected?
-Which events are missed or incorrectly predicted?
-How does performance change with different prompting or segmentation strategies?

The generated outputs may be quite descriptive. While this can introduce noise, it also provides an opportunity to infer potential events through reasoning over these descriptions.

Therefore, one possible strategy is to first generate scene-level descriptions using a vision-language model (VLM), and then further process these descriptions using a language model (LLM) to extract structured events.

The choice of strategy is left to you. You are expected to design and implement an approach that produces meaningful event representations from the video.

#Retrieve all of the salient events of the video

Old prompting strategy, not used in the final pipeline.

In [ ]:
def detect_events_old(frames, include_timestamps=False, prompt_id=1):
    if prompt_id == 1:
        prompt = (
            "You are analyzing chronological frames sampled from ONE tourist landmark video of the Statue of Liberty. "
            "Describe ONLY what is directly visible. Do NOT invent elements that are not present. "
            "Focus on identifying only the salient events. "
            "Return concise chronological events."
        )
    elif prompt_id == 2:
        prompt = (
            "You are analyzing chronological frames sampled from ONE aerial drone flight video. "
            "The footage is shot from above and shows landscapes, fields, water, trees, and a small aircraft. "
            "Describe ONLY what is directly visible from the aerial perspective. "
            "Do NOT invent people, buildings, or ground-level scenes. "
            "Focus on camera movement, landscape changes, and aircraft behaviour. "
            "Return concise chronological events."
        )
    elif prompt_id == 3:
        prompt = (
            "You are analyzing chronological frames sampled from ONE extreme sports video showing BMX cycling. "
            "The footage shows a cyclist performing tricks, navigating obstacles, and racing through an urban course. "
            "Describe ONLY what is directly visible. Do NOT invent crowds, commentators, or off-screen elements. "
            "Focus on the cyclist's actions, specific obstacles, and course landmarks such as ramps, walls, and banners. "
            "Return concise chronological events."
        )
    elif prompt_id == 4:
        prompt = (
            "You are analyzing chronological frames sampled from ONE outdoor ground-level video. "
            "The footage is filmed from a camera placed low to the ground and involves vehicles and people interacting with the camera. "
            "Describe ONLY what is directly visible. Do NOT invent indoor scenes or unrelated elements. "
            "Focus on vehicles, people, camera movement, and any objects entering or leaving the frame. "
            "Return concise chronological events."
        )

    if include_timestamps:
        prompt += " Format exactly: Salient event X: description"
    else:
        prompt += " Format exactly: Salient event X: description"

    messages = [{"role": "user", "content": [{"type": "video", "data": frames}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=220, do_sample=False, num_beams=4, repetition_penalty=1.15, no_repeat_ngram_size=3)
    txt = processor.batch_decode(out, skip_special_tokens=True)[0]
    return txt

### Improved Prompt Engineering Strategy
To reduce hallucinations, we implement the following changes:
- **Negative Constraints**: Explicitly forbidding the inference of off-screen actions.
- **Visual Grounding**: Requiring the model to focus only on distinctive visual changes.
- **Ambiguity Handling**: Instructing the model to ignore frames that are low-quality or unrecognizable.

In [ ]:
def detect_events(frames, prompt_id=1):
    """
    Optimized event detection using grounded prompt engineering.
    """
    # Base strategy for all prompts
    base_strategy = (
        "Task: Extract a chronological list of salient physical events from the provided video frames.\n"
        "Constraint 1: DESCRIPTION ONLY. Only describe actions clearly visible in the frames. If the movement is ambiguous, omit it.\n"
        "Constraint 2: NO INFERENCE. Do not assume what happened between frames or why a person is doing something. Focus on the 'what', not the 'why'.\n"
        "Constraint 3: ZERO HALLUCINATION. Do not mention objects, people, or text not explicitly visible in the pixels.\n"
    )

    prompts = {
        1: "Video Context: Tourist landmark (Statue of Liberty). Focus on camera movement (pan/zoom) and group interactions with the monument.",
        2: "Video Context: Aerial drone footage. Focus on geographical features and the drone's flight path/altitude changes.",
        3: "Video Context: BMX extreme sports. Focus on specific tricks (jumps, spins) and contact with obstacles (ramps, walls).",
        4: "Video Context: Ground-level vehicle/pedestrian interaction. Focus on objects entering/exiting the frame and proximity to the lens.",    }

    selected_context = prompts.get(prompt_id, "General video analysis.")

    final_prompt = f"{base_strategy}\nContext: {selected_context}\nFormat: Salient event X: [Short Description]"

    messages = [{"role": "user", "content": [{"type": "video", "data": frames}, {"type": "text", "text": final_prompt}]}]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)

    out = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=0.0, repetition_penalty=1.2)
    return processor.batch_decode(out, skip_special_tokens=True)[0]

In [ ]:
def detect_events_few_shot(frames, prompt_id=1, few_shot_examples=None):
    """
    Optimized event detection using grounded prompt engineering with few-shot examples.
    """
    # Base strategy for all prompts
    base_strategy = (
        "Task: Extract a chronological list of salient physical events from the provided video frames.\n"
        "Constraint 1: DESCRIPTION ONLY. Only describe actions clearly visible in the frames. If the movement is ambiguous, omit it.\n"
        "Constraint 2: NO INFERENCE. Do not assume what happened between frames or why a person is doing something. Focus on the 'what', not the 'why'.\n"
        "Constraint 3: ZERO HALLUCINATION. Do not mention objects, people, or text not explicitly visible in the pixels.\n"
    )

    prompts = {
        1: "Video Context: Tourist landmark (Statue of Liberty). Focus on camera movement (pan/zoom) and group interactions with the monument.",
        2: "Video Context: Aerial drone footage. Focus on geographical features and the drone's flight path/altitude changes.",
        3: "Video Context: BMX extreme sports. Focus on specific tricks (jumps, spins) and contact with obstacles (ramps, walls).",
        4: "Video Context: Ground-level vehicle/pedestrian interaction. Focus on objects entering/exiting the frame and proximity to the lens.",
        5: "Video Context: Toy airplane aerial footage. Focus on the toy airplane's flight path, maneuvers, and interaction with the environment (ground, trees)."
    }

    selected_context = prompts.get(prompt_id, "General video analysis.")

    few_shot_string = ""
    if few_shot_examples:
        few_shot_string += "\nHere are some examples:\n"
        for i, example in enumerate(few_shot_examples):
            few_shot_string += f"Example {i+1} Scenario: {example['scenario']}\n"
            few_shot_string += f"Example {i+1} Events: {example['events']}\n\n"

    final_prompt = f"{base_strategy}\nContext: {selected_context}\n{few_shot_string}Format: Salient event X: [Short Description]"

    messages = [{"role": "user", "content": [{"type": "video", "data": frames}, {"type": "text", "text": final_prompt}]}]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)

    out = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=0.0, repetition_penalty=1.2)
    return processor.batch_decode(out, skip_special_tokens=True)[0]

In [ ]:
toy_airplane_few_shot_examples = [
    {
        "scenario": "A toy airplane flying low over a grassy field, performing a loop-the-loop.",
        "events": "Salient event 1: Toy airplane takes off from the ground.\nSalient event 2: Toy airplane flies upwards, beginning a loop.\nSalient event 3: Toy airplane completes a loop-the-loop in the air.\nSalient event 4: Toy airplane descends towards the field.\nSalient event 5: Toy airplane lands gently on the grass."
    },
    {
        "scenario": "A toy airplane soaring above trees and then disappearing behind them.",
        "events": "Salient event 1: Toy airplane is visible flying above a line of trees.\nSalient event 2: Toy airplane continues flying, getting closer to the trees.\nSalient event 3: Toy airplane begins to descend slightly towards the tree line.\nSalient event 4: Toy airplane disappears behind the trees."
    },
    {
        "scenario": "A toy airplane making a sharp turn in the sky, with some clouds in the background.",
        "events": "Salient event 1: Toy airplane flies straight across the sky.\nSalient event 2: Toy airplane executes a sharp turn to the left.\nSalient event 3: Toy airplane continues flying in the new direction.\nSalient event 4: Clouds are visible moving slowly in the background."
    }
]

In [ ]:

def add_timestamps_to_events(events, meta):
    if not events: return []
    times=[m["time"] for m in meta]
    total=max(times) if times else 0
    n=len(events)
    stamped=[]
    for i,e in enumerate(events):
        start = total*i/n
        end = total*(i+1)/n
        def fmt(t):
            m=int(t//60); s=int(t%60)
            return f"{m:02d}:{s:02d}"
        stamped.append(f"{e} ({fmt(start)} - {fmt(end)})")
    return stamped


#Parser to extract events and use word embedding similarity to compare them with your manual ground truth

In [ ]:

import re
from sentence_transformers import SentenceTransformer, util



def parse_vlm_output(text):
    text=text.replace("Salent","Salient").replace("Salentevent","Salient event").replace("SalientEvent","Salient event")
    pats = re.findall(r'(?:Salient event\s*\d+[:.]?|\d+[.)])\s*(.*?)(?=(?:\n(?:Salient event\s*\d+|\d+[.)]))|$)', text, flags=re.I|re.S)
    events=[]
    for p in pats:
        e=re.sub(r'\s+',' ',p).strip(' .-')
        e=re.sub(r'\(\d\d:\d\d.*?\)','',e).strip()
        if len(e)>8 and e.lower() not in [x.lower() for x in events]:
            events.append(e)
    return events

def evaluate(manual, predicted, threshold=0.30):
    # Initialize the model (using a standard small/fast model as an example)
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    emb1=embedding_model.encode(manual, convert_to_tensor=True, normalize_embeddings=True)
    emb2=embedding_model.encode(predicted, convert_to_tensor=True, normalize_embeddings=True)
    sim=util.cos_sim(emb1, emb2)
    hit=0
    for i,m in enumerate(manual):
        j=int(sim[i].argmax())
        score=float(sim[i][j])
        status="MATCH" if score>=threshold else "MISSED"
        print(f"{status}: {m[:55]} -> {predicted[j][:55]} (Score {score:.2f})")
        if score>=threshold: hit+=1
    print(f"Coverage: {hit}/{len(manual)} = {100*hit/len(manual):.1f}%")


In [ ]:
video_frames, meta = sample_frames(video_path, num_frames=24)

print("--- Running Step 3a: Event Only (V2) ---")
# For video_21, use prompt_id=5 (toy airplane aerial footage)
raw_3a = detect_events(video_frames, prompt_id=2)
parsed_events_3a = parse_vlm_output(raw_3a)

print("\n--- Running Step 3b: Event + Timestamps (V2) ---")
# For video_21, use prompt_id=5 (toy airplane aerial footage)
raw_3b = detect_events(video_frames, prompt_id=2)
parsed_events_3b = parse_vlm_output(raw_3b)
parsed_events_3b = add_timestamps_to_events(parsed_events_3b, meta)

print("\n--- Output 3a ---")
for i,e in enumerate(parsed_events_3a,1):
    print(f"{i}. {e}")

print("\n--- Output 3b ---")
for i,e in enumerate(parsed_events_3b,1):
    print(f"{i}. {e}")

print("\n--- Evaluating 3a ---")
evaluate(manual_annotations, parsed_events_3a, threshold=0.30)

print("\n--- Evaluating 3b ---")
evaluate(manual_annotations, [re.sub(r'\(.*?\)','',x).strip() for x in parsed_events_3b], threshold=0.30)


# Step 4 — Moment Retrieval using Moment-DETR + CLIP Features

This section performs temporal moment retrieval using the event descriptions generated by SmolVLM2 in Step 3.

### What this section does
1. Loads a pre-trained Moment-DETR model
2. Extracts video features using CLIP
3. Uses SmolVLM2 event descriptions as retrieval queries
4. Tests multiple query variations for stronger alignment
5. Retrieves temporal segments from the video
6. Merges overlapping intervals and filters low-confidence predictions
7. Returns timestamps in the format:
```
[start_time, end_time]
```

In [ ]:
# ============================================================
# STEP 4: MOMENT RETRIEVAL
# ============================================================

# Install additional dependencies if needed
# !pip install git+https://github.com/jayleicn/moment_detr.git
# !pip install open_clip_torch

import os
import cv2
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# 1. Load CLIP model for feature extraction
# ------------------------------------------------------------

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# ------------------------------------------------------------
# 2. Extract temporal video features
# ------------------------------------------------------------

def extract_video_features(video_path, sample_rate=12):
    """
    Extract CLIP frame embeddings from the video.
    Returns:
        features: Tensor [num_frames, feature_dim]
        timestamps: list of timestamps for each sampled frame
    """

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    features = []
    timestamps = []

    frame_idx = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx % sample_rate == 0:

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(rgb)

            inputs = clip_processor(images=image, return_tensors="pt").to(device)

            with torch.no_grad():
                image_features = clip_model.get_image_features(**inputs)

            # Access the pooler_output tensor from the model output object
            embeddings = image_features.pooler_output
            # Now perform the normalization on the tensor
            image_features = embeddings / embeddings.norm(dim=-1, keepdim=True)

            features.append(image_features.cpu())
            timestamps.append(frame_idx / fps)

        frame_idx += 1

    cap.release()

    if len(features) == 0:
        raise ValueError("No features extracted from video.")

    features = torch.cat(features, dim=0)

    return features, timestamps


In [ ]:
# ------------------------------------------------------------
# 3. Create query variations
# ------------------------------------------------------------

def build_query_variations(event_text):

    variations = [
        event_text,
        f"A person is {event_text}",
        f"The video shows {event_text}",
        f"An action where {event_text}",
        f"A scene containing {event_text}"
    ]

    return variations

In [ ]:
# ------------------------------------------------------------
# 4. Encode text queries using CLIP
# ------------------------------------------------------------

def encode_text_queries(queries):
    # Ensure queries is a list of strings
    inputs = clip_processor(text=queries, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)

    # Access the tensor from the BaseModelOutputWithPooling object
    embeddings = outputs.pooler_output
    # Perform the normalization
    text_features = embeddings / embeddings.norm(dim=-1, keepdim=True)

    return text_features.cpu()

In [ ]:
# ------------------------------------------------------------
# 5. Retrieve candidate temporal moments
# ------------------------------------------------------------

def retrieve_moments(video_features,
                     timestamps,
                     query_text,
                     top_k=5,
                     window_size=3):

    query_variations = build_query_variations(query_text)
    text_features = encode_text_queries(query_variations)
    all_results = []

    for q_idx in range(len(query_variations)):
        q_feat = text_features[q_idx].unsqueeze(0)
        similarities = util.cos_sim(q_feat, video_features)[0]
        top_indices = torch.topk(similarities, k=min(top_k, len(similarities))).indices
        for idx in top_indices:
            idx = idx.item()
            start_idx = max(0, idx - window_size)
            end_idx = min(len(timestamps)-1, idx + window_size)
            start_time = timestamps[start_idx]
            end_time = timestamps[end_idx]
            score = similarities[idx].item()
            all_results.append({
                "query": query_variations[q_idx],
                "start": float(start_time),
                "end": float(end_time),
                "score": float(score)
            })

    return all_results



In [ ]:
# ------------------------------------------------------------
# 6. Post-processing:
#    Merge overlaps + confidence filtering
# ------------------------------------------------------------

def merge_overlapping_segments(segments,
                               overlap_threshold=0.5,
                               score_threshold=0.20):

    # Filter low-confidence predictions
    segments = [s for s in segments if s["score"] >= score_threshold]

    if len(segments) == 0:
        return []

    # Sort by start time
    segments = sorted(segments, key=lambda x: x["start"])

    merged = [segments[0]]

    for current in segments[1:]:

        prev = merged[-1]

        inter_start = max(prev["start"], current["start"])
        inter_end = min(prev["end"], current["end"])

        intersection = max(0, inter_end - inter_start)

        union = max(prev["end"], current["end"]) - min(prev["start"], current["start"])

        overlap = intersection / union if union > 0 else 0

        if overlap >= overlap_threshold:

            prev["end"] = max(prev["end"], current["end"])
            prev["score"] = max(prev["score"], current["score"])

        else:
            merged.append(current)

    return merged

In [ ]:
# ------------------------------------------------------------
# 7. Full retrieval pipeline
# ------------------------------------------------------------

def run_moment_retrieval(video_path,
                         event_descriptions,
                         top_k=5):

    video_features, timestamps = extract_video_features(video_path)

    retrieval_results = {}

    for event_text in event_descriptions:

        print(f"Retrieving moments for: {event_text}")

        raw_segments = retrieve_moments(
            video_features,
            timestamps,
            event_text,
            top_k=top_k
        )

        final_segments = merge_overlapping_segments(raw_segments)

        # Format output for IoU evaluation
        timestamp_pairs = [
            [round(seg["start"], 2), round(seg["end"], 2)]
            for seg in final_segments
        ]

        retrieval_results[event_text] = timestamp_pairs

        print("Retrieved Segments:")
        for seg in timestamp_pairs:
            print(seg)

    return retrieval_results


In [ ]:
# ------------------------------------------------------------
# 8. Use Step 3 SmolVLM2 events as retrieval queries
# ------------------------------------------------------------

# Example:
# parsed_events = parse_vlm_output(raw_3a)

if isinstance(parsed_events_3b, list):
    event_queries = parsed_events_3b
else:
    event_queries = [parsed_events_3b]

retrieved_event_segments = run_moment_retrieval(
    video_path,
    event_queries,
    top_k=5
)

print("==============================")
print("FINAL MOMENT RETRIEVAL OUTPUT")
print("==============================")

for event_name, timestamps in retrieved_event_segments.items():
    print(f"EVENT: {event_name}")
    print(timestamps)

# Output format:
# {
#   "person opens the door": [[12.4, 18.1], [32.0, 36.8]],
#   "person sits on chair": [[41.2, 48.0]]
# }

# Step 4b — Second Retrieval Model: Adaptive Threshold Peak Detection

The first model (Step 4) scores **individual frames** independently and pads a fixed
window of fixed size around the top-k highest-scoring frames. This second model uses
an **adaptive threshold peak detection** strategy: it computes frame-level CLIP
similarity scores, then identifies contiguous regions where the similarity stays
consistently above an adaptive threshold (mean + 0.5 × std of the score distribution).
These regions are treated as candidate segments directly, without any fixed window padding.

Key differences from Step 4:
- **No fixed window size**: segment boundaries are determined by where similarity
  drops below the adaptive threshold, not by padding a fixed number of frames.
- **Contiguous region detection**: instead of finding isolated top-k frames, it
  finds continuous high-similarity intervals, which better captures events that
  span multiple frames with sustained visual relevance.
- **Adaptive threshold**: the threshold adapts to each query–video pair, making
  it more robust to videos with different overall similarity distributions.



In [ ]:
import torch
import numpy as np


def run_adaptive_threshold_retrieval(video_path, event_queries, top_k=3, score_threshold=0.15):
    """
    Second retrieval model: adaptive threshold peak detection over CLIP frame similarities.

    Strategy:
      1. Extract CLIP frame features (reuses extract_video_features from Step 4).
      2. Compute cosine similarity between the text query and every frame.
      3. Set an adaptive threshold = mean(similarities) + 0.5 * std(similarities).
         This adapts to each query-video pair rather than using a fixed cutoff.
      4. Find contiguous frame runs that stay above the threshold.
         Each run becomes a candidate temporal segment.
      5. Sort candidates by peak score, keep top_k, merge overlapping intervals.
      6. Fallback: if no segment passes the threshold, return a small window
         around the single highest-scoring frame.

    Differs from Step 4 (fixed-window CLIP) in that:
      - Segment boundaries come from the similarity signal itself, not fixed padding.
      - Contiguous high-similarity regions are captured as whole intervals.
      - The threshold is query-adaptive, not globally fixed.

    Args:
        video_path    : path to the video file
        event_queries : list of event description strings (from SmolVLM2 Step 3)
        top_k         : max number of segments to keep per query
        score_threshold: hard minimum score; adaptive threshold is always >= this

    Returns:
        dict {event_text: [[start, end], ...]}
    """
    # Reuse feature extractor from Step 4
    video_features, timestamps = extract_video_features(video_path)
    video_duration = timestamps[-1] if timestamps else 1.0

    retrieval_results = {}

    for event_text in event_queries:
        print(f"[AdaptiveThreshold] Retrieving: '{event_text[:60]}'")

        # --- Step 1: Encode text query with CLIP ---
        text_inputs = clip_processor(
            text=[event_text], return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            text_out = clip_model.get_text_features(**text_inputs)
        text_features = text_out.pooler_output
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # --- Step 2: Frame-level cosine similarity ---
        with torch.no_grad():
            similarities = torch.nn.functional.cosine_similarity(
                text_features.cpu(), video_features, dim=-1
            )
        sim_np = similarities.numpy()

        # --- Step 3: Adaptive threshold ---
        # Threshold adapts to the distribution of scores for this specific query,
        # rather than using a global fixed cutoff.
        adaptive_threshold = np.mean(sim_np) + 0.5 * np.std(sim_np)
        threshold = max(adaptive_threshold, score_threshold)

        # --- Step 4: Find contiguous high-similarity regions ---
        # Each contiguous run of frames above threshold becomes one candidate segment.
        peaks = []
        inside_segment = False
        start_idx = 0

        for idx, score in enumerate(sim_np):
            if score >= threshold and not inside_segment:
                start_idx = idx
                inside_segment = True
            elif score < threshold and inside_segment:
                end_idx = idx - 1
                peaks.append({
                    "start": timestamps[start_idx],
                    "end":   timestamps[end_idx],
                    "score": float(np.max(sim_np[start_idx:idx]))
                })
                inside_segment = False

        # Close any segment still open at the end of the video
        if inside_segment:
            peaks.append({
                "start": timestamps[start_idx],
                "end":   timestamps[-1],
                "score": float(np.max(sim_np[start_idx:]))
            })

        # --- Step 5: Sort by score, keep top_k, merge overlaps ---
        peaks = sorted(peaks, key=lambda x: x["score"], reverse=True)[:top_k]
        merged = merge_overlapping_segments(peaks, overlap_threshold=0.4, score_threshold=0.0)
        timestamp_pairs = [[round(s["start"], 2), round(s["end"], 2)] for s in merged]

        # --- Step 6: Fallback if no segment found ---
        # If nothing passes the threshold, return a small window around
        # the single highest-scoring frame so IoU evaluation never gets an empty result.
        if not timestamp_pairs and len(timestamps) > 0:
            best_idx = int(similarities.argmax())
            fallback_start = max(0.0, timestamps[best_idx] - 2.0)
            fallback_end   = min(video_duration, timestamps[best_idx] + 2.0)
            timestamp_pairs = [[round(fallback_start, 2), round(fallback_end, 2)]]

        retrieval_results[event_text] = timestamp_pairs
        print(f"  -> {timestamp_pairs}")

    return retrieval_results


print("Adaptive threshold retrieval function defined.")


In [ ]:
retrieved_adaptive_segments = run_adaptive_threshold_retrieval(
    video_path, event_queries, top_k=3
)

print("=====================================")
print("ADAPTIVE THRESHOLD RETRIEVAL OUTPUT")
print("=====================================")
for event_name, ts in retrieved_adaptive_segments.items():
    print(f"EVENT: {event_name}")
    print(ts)

# Step 5 — IoU Evaluation

Evaluates the temporal accuracy of both retrieval models against the manual annotations.
For each annotated event the evaluator:
1. Finds the best-matching retrieved event via sentence-embedding similarity on descriptions
2. Computes IoU between the predicted and ground-truth time intervals
3. Reports per-event IoU, mean IoU, Recall@0.5, and a failure analysis


In [ ]:
# ============================================================
# STEP 5: IoU HELPER FUNCTIONS
# ============================================================

import re
from sentence_transformers import SentenceTransformer, util as st_util


def load_annotations_with_timestamps(file_path):
    """
    Parses annotation lines of the form:
        Description, MM:SS - MM:SS, score
    Returns list of dicts: {description, start, end, score}
    """
    annotations = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 3:
                continue
            description = parts[0].strip()
            time_str    = parts[1].strip()
            score_str   = parts[2].strip()
            match = re.match(r'(\d+):(\d+)\s*-\s*(\d+):(\d+)', time_str)
            if not match:
                continue
            m1, s1, m2, s2 = match.groups()
            start = int(m1) * 60 + int(s1)
            end   = int(m2) * 60 + int(s2)
            try:
                score = int(score_str)
            except ValueError:
                score = 0
            annotations.append({'description': description,
                                 'start': float(start), 'end': float(end),
                                 'score': score})
    return annotations


def compute_iou(pred_start, pred_end, gt_start, gt_end):
    inter = max(0.0, min(pred_end, gt_end) - max(pred_start, gt_start))
    union = max(0.0, max(pred_end, gt_end) - min(pred_start, gt_start))
    return inter / union if union > 0 else 0.0


_iou_embed_model = None

def evaluate_iou(annotations, retrieval_results, iou_threshold=0.5):
    """
    Matches each annotation to the best retrieved segment by embedding similarity,
    then computes IoU on timestamps.
    Returns: (results list, mean_iou float)
    """
    global _iou_embed_model
    if _iou_embed_model is None:
        _iou_embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    gt_descs = [a['description'] for a in annotations]
    ret_keys = list(retrieval_results.keys())
    if not ret_keys:
        return [], 0.0

    gt_embs  = _iou_embed_model.encode(gt_descs,  convert_to_tensor=True, normalize_embeddings=True)
    ret_embs = _iou_embed_model.encode(ret_keys,   convert_to_tensor=True, normalize_embeddings=True)
    sim      = st_util.cos_sim(gt_embs, ret_embs)

    results = []
    for i, ann in enumerate(annotations):
        best_ret_idx = int(sim[i].argmax())
        desc_sim     = float(sim[i][best_ret_idx])
        matched_key  = ret_keys[best_ret_idx]
        segments     = retrieval_results[matched_key]

        if not segments:
            best_iou, best_seg = 0.0, None
        else:
            ious     = [compute_iou(s[0], s[1], ann['start'], ann['end']) for s in segments]
            best_iou = max(ious)
            best_seg = segments[ious.index(best_iou)]

        results.append({
            'gt_description':  ann['description'],
            'gt_start':        ann['start'],
            'gt_end':          ann['end'],
            'subjectivity':    ann['score'],
            'matched_event':   matched_key,
            'desc_similarity': desc_sim,
            'best_segment':    best_seg,
            'iou':             best_iou,
            'hit':             best_iou >= iou_threshold
        })

    mean_iou = sum(r['iou'] for r in results) / len(results)
    return results, mean_iou


def print_iou_report(results, mean_iou, model_label='', iou_threshold=0.5):
    print(f"\n{'='*65}")
    print(f"  IoU REPORT — {model_label}")
    print(f"{'='*65}")
    for i, r in enumerate(results):
        seg_str = (f"{r['best_segment'][0]:.1f}s-{r['best_segment'][1]:.1f}s"
                   if r['best_segment'] else 'none')
        status = '\u2713' if r['hit'] else '\u2717'
        print(f"{i+1:>2}. {status} IoU={r['iou']:.2f}  "
              f"GT=[{r['gt_start']:.0f}s-{r['gt_end']:.0f}s]  Pred=[{seg_str}]")
        print(f"    GT : {r['gt_description'][:60]}")
        print(f"    Ret: {r['matched_event'][:60]}  (sim={r['desc_similarity']:.2f})")
    hits = sum(1 for r in results if r['hit'])
    print(f"\nMean IoU   : {mean_iou:.3f}")
    print(f"Recall@{iou_threshold} : {hits}/{len(results)} = {100*hits/max(len(results),1):.1f}%")
    failures = [r for r in results if not r['hit']]
    print(f"\n--- Failure Analysis ---")
    if not failures:
        print('  All events retrieved above threshold.')
    else:
        for r in failures:
            if r['iou'] < 0.05:
                reason = 'No temporal overlap — wrong region retrieved entirely.'
            elif r['desc_similarity'] < 0.4:
                reason = 'Poor query match — description not aligned with retrieved event.'
            else:
                reason = 'Partial overlap — correct region but imprecise boundaries.'
            print(f"  [{r['subjectivity']:+d}] '{r['gt_description'][:55]}' "
                  f"IoU={r['iou']:.2f} -> {reason}")


print('IoU helper functions defined.')


In [ ]:
# ============================================================
# STEP 5: RUN IoU EVALUATION — video_20
# ============================================================

ANN1_PATH = '/content/18-SumMe-video_20-annotator1.txt'
ANN2_PATH = '/content/18-SumMe-video_20-annotator2.txt'

annotations_1 = load_annotations_with_timestamps(ANN2_PATH)
print(f'Loaded {len(annotations_1)} annotations.')

# --- Model 1: Frame-level CLIP (Step 4) ---
# Strip inline timestamps appended by add_timestamps_to_events before matching
clean_queries = [re.sub(r'\(.*?\)', '', q).strip() for q in event_queries]
iou_clip, mean_iou_clip = evaluate_iou(annotations_1, retrieved_event_segments)
print_iou_report(iou_clip, mean_iou_clip, model_label='Frame-level CLIP (Step 4)')

# --- Model 2: Adaptive Threshold Peak Detection (Step 4b) ---
iou_adaptive, mean_iou_adaptive = evaluate_iou(annotations_1, retrieved_adaptive_segments)
print_iou_report(iou_adaptive, mean_iou_adaptive, model_label='Adaptive Threshold (Step 4b)')


In [ ]:
# ============================================================
# STEP 5: SIDE-BY-SIDE MODEL COMPARISON
# ============================================================

print(f"{'='*65}")
print(f"  MODEL COMPARISON — video_20")
print(f"{'='*65}")
print(f"{'GT Event':<42} {'CLIP':>6} {'Adaptive':>9}")
print('-'*60)
for r_c, r_a in zip(iou_clip, iou_adaptive):
    label = r_c['gt_description'][:42]
    cm = '✓' if r_c['hit'] else '✗'
    am = '✓' if r_a['hit'] else '✗'
    print(f"{label:<42} {r_c['iou']:>4.2f}{cm}  {r_a['iou']:>4.2f}{am}")
print('-'*60)
print(f"{'Mean IoU':<42} {mean_iou_clip:>6.3f}  {mean_iou_adaptive:>6.3f}")
better = 'Adaptive Threshold' if mean_iou_adaptive > mean_iou_clip else 'Frame-level CLIP'
print(f"Better overall: {better}")
print("Key differences:")
print("  Frame-level CLIP    : scores individual frames, pads a fixed window around top-k hits.")
print("  Adaptive Threshold  : finds contiguous high-similarity regions using a query-adaptive")
print("  threshold (mean + 0.5*std), no fixed window size.")


In [ ]:
# ============================================================
# FULL PIPELINE — ALL 4 VIDEOS
# ============================================================

GROUP_ID = '18'
DATASET  = 'SumMe'

VIDEOS = [
    {'stem': 'video_20', 'label': 'video_20', 'prompt_id': 1},
    {'stem': 'video_21', 'label': 'video_21', 'prompt_id': 2},
    {'stem': 'video_22', 'label': 'video_22', 'prompt_id': 3},
    {'stem': 'video_23', 'label': 'video_23', 'prompt_id': 4},
]

all_results = {}

for video_cfg in VIDEOS:
    stem      = video_cfg['stem']
    label     = video_cfg['label']
    prompt_id = video_cfg['prompt_id']
    vpath     = f'/content/{stem}.mp4'
    ann1      = f'/content/{GROUP_ID}-{DATASET}-{label}-annotator1.txt'
    ann2      = f'/content/{GROUP_ID}-{DATASET}-{label}-annotator2.txt'

    print('\n' + '='*65)
    print(f'  VIDEO: {stem}')
    print('='*65)

    if not os.path.exists(ann1):
        print(f'  WARNING: {ann1} not found — skipping.')
        continue

    annotations_v = load_annotations_with_timestamps(ann1)
    ann_descs     = [a['description'] for a in annotations_v]
    print(f'  Annotator 1: {len(annotations_v)} events')

    if os.path.exists(ann2):
        anns2 = load_annotations_with_timestamps(ann2)
        print(f'  Annotator 2: {len(anns2)} events')
        kappa, n_matched = compute_cohen_kappa(ann1, ann2)
        if kappa is not None:
            print(f"  Cohen's Kappa (Ann1 vs Ann2): {kappa:.3f} ({n_matched} matched pairs)")
    else:
        print(f'  Annotator 2 file not found — skipping Kappa.')

    # Step 3
    print(f'  [Step 3] Running SmolVLM2 (prompt_id={prompt_id})...')
    vframes, vmeta = sample_frames(vpath, num_frames=24)

    # Fix: Removed include_timestamps argument as it is not in the new detect_events
    raw_a = detect_events(vframes, prompt_id=prompt_id)
    ev_a  = parse_vlm_output(raw_a)

    raw_b = detect_events(vframes, prompt_id=prompt_id)
    ev_b  = parse_vlm_output(raw_b)
    ev_b  = add_timestamps_to_events(ev_b, vmeta)

    eq    = ev_b if ev_b else ev_a
    print(f'  Detected {len(eq)} events.')

    print('  [Step 3 Eval] Coverage vs annotator 1:')
    evaluate(ann_descs, [re.sub(r'\(.*?\)', '', e).strip() for e in eq], threshold=0.30)

    # Step 4
    print(f'  [Step 4] Frame-level CLIP retrieval...')
    ret_clip = run_moment_retrieval(vpath, eq, top_k=5)

    # Step 4b
    print(f'  [Step 4b] Adaptive threshold retrieval...')
    ret_sw = run_adaptive_threshold_retrieval(vpath, eq, top_k=3)

    # Step 5
    iou_c, miou_c = evaluate_iou(annotations_v, ret_clip)
    iou_s, miou_s = evaluate_iou(annotations_v, ret_sw)
    print_iou_report(iou_c, miou_c, model_label=f'CLIP — {stem}')
    print_iou_report(iou_s, miou_s, model_label=f'Adaptive Threshold — {stem}')

    better = 'Adaptive Threshold' if miou_s > miou_c else 'CLIP'
    print(f'  -> Better: {better} (CLIP={miou_c:.3f}, Adaptive={miou_s:.3f})')

    all_results[stem] = {
        'event_queries':  eq,
        'retrieved_clip': ret_clip,
        'retrieved_sw':   ret_sw,
        'annotations':    annotations_v,
        'iou_clip':       iou_c,
        'iou_sw':         iou_s,
        'mean_clip':      miou_c,
        'mean_sw':        miou_s,
        'video_path':     vpath,
    }

print('\n' + '='*65)
print('  SUMMARY ACROSS ALL VIDEOS')
print('='*65)
print(f"{'Video':<12} {'CLIP mIoU':>10} {'Adaptive mIoU':>14} {'Better':>18}")
print('-'*56)
for s, res in all_results.items():
    b = 'Adaptive Threshold' if res['mean_sw'] > res['mean_clip'] else 'CLIP'
    print(f"{s:<12} {res['mean_clip']:>10.3f} {res['mean_sw']:>14.3f} {b:>18}")

# **Video Summary**

In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os

def generate_video_summary(video_path, segments, output_filename="summary.mp4"):
    if not segments:
        print(f"No segments found for {video_path}. Skipping.")
        return

    video = VideoFileClip(video_path)
    clips = []

    # 1. Sort segments by start time
    sorted_segments = sorted(segments, key=lambda x: x[0])

    # 2. Refined Filtering: Remove extremely short clips that cause 'stuttering'
    min_duration = 1.0 # Ignore anything shorter than 1 second

    for start, end in sorted_segments:
        duration = end - start
        if duration >= min_duration:
            end = min(end, video.duration)
            start = max(0, start)
            clips.append(video.subclip(start, end))

    if clips:
        print(f"Creating refined summary: {output_filename} with {len(clips)} segments.")
        final_summary = concatenate_videoclips(clips)
        final_summary.write_videofile(output_filename, codec="libx264", audio_codec="aac", temp_audiofile='temp-audio.m4a', remove_temp=True)

    video.close()

# --- IMPROVED BATCH PROCESSING WITH GAP FILLING ---
for video_id, res in all_results.items():
    print(f"\nProcessing Refined Summary for: {video_id}")

    raw_segments = []
    for query_text, segments in res['retrieved_sw'].items():
        for s in segments:
            raw_segments.append({"start": s[0], "end": s[1], "score": 1.0})

    # Strategy: Use a very low overlap threshold (0.05) to be more aggressive
    # and consider segments 'overlapping' if they are nearly touching
    merged = merge_overlapping_segments(raw_segments, overlap_threshold=0.05)

    # Further refinement: merge segments that are separated by less than 1.5 seconds
    # to create continuous sequences instead of jump-cuts
    final_timestamps = []
    if merged:
        merged = sorted(merged, key=lambda x: x['start'])
        current_start = merged[0]['start']
        current_end = merged[0]['end']

        for next_seg in merged[1:]:
            if next_seg['start'] - current_end < 1.5: # Gap tolerance
                current_end = max(current_end, next_seg['end'])
            else:
                final_timestamps.append([current_start, current_end])
                current_start = next_seg['start']
                current_end = next_seg['end']
        final_timestamps.append([current_start, current_end])

    out_name = f"Group18_{video_id}_Summary_Refined.mp4"
    generate_video_summary(
        video_path=res['video_path'],
        segments=final_timestamps,
        output_filename=out_name
    )

### Step 6: Targeted Improvement for Each Video
Because each video has different complexity, we will now override the parameters specifically for the videos that aren't performing well.
- **Video 20/23**: We will reduce `top_k` to force a shorter summary.
- **Video 22**: We will lower the `score_threshold` and increase `window_size` to capture missed jumps.

In [ ]:
def generate_targeted_summaries(all_results):
    # Configuration overrides for problematic videos
    overrides = {
        'video_20': {'top_k': 2, 'score_threshold': 0.25},
        'video_23': {'top_k': 2, 'score_threshold': 0.25},
        'video_22': {'top_k': 8, 'score_threshold': 0.12, 'window_size': 5},
    }

    for v_id in ['video_20', 'video_22', 'video_23']:
        res = all_results[v_id]
        cfg = overrides.get(v_id)

        print(f"\n>>> Re-processing {v_id} with optimized targeted parameters: {cfg}")

        # OPTIMIZATION: Extract video features ONCE per video, not once per query
        video_features, timestamps = extract_video_features(res['video_path'])

        raw_segments = []
        for query in res['event_queries']:
            # Use the pre-extracted features
            segs = retrieve_moments(
                video_features,
                timestamps,
                query,
                top_k=cfg['top_k'],
                window_size=cfg.get('window_size', 3)
            )
            raw_segments.extend(segs)

        # Process and save as before
        merged = merge_overlapping_segments(raw_segments, score_threshold=cfg['score_threshold'])

        final_ts = []
        if merged:
            merged = sorted(merged, key=lambda x: x['start'])
            curr_s, curr_e = merged[0]['start'], merged[0]['end']
            for n in merged[1:]:
                if n['start'] - curr_e < 1.5: curr_e = max(curr_e, n['end'])
                else:
                    final_ts.append([curr_s, curr_e])
                    curr_s, curr_e = n['start'], n['end']
            final_ts.append([curr_s, curr_e])

        generate_video_summary(res['video_path'], final_ts, f"Group18_{v_id}_FIXED.mp4")

generate_targeted_summaries(all_results)